## Clean NIFD data and select latest visits

Created by: Varuna & Sahana

Date: 05/07/2025

In [1]:
import pandas as pd
import json

In [2]:
with open("/projectnb/vkolagrp/datasets/NIFD/data_dictionaries/NIFD_dictionary.json", 'r') as json_file:
    data_dict = json.load(json_file)
nifd = pd.read_csv('/projectnb/vkolagrp/datasets/NIFD/csv/raw/NIFD_Clinical_Data_20200724_updated.csv')

In [3]:
negative_rows = nifd[nifd.select_dtypes(include=['number']).lt(0).any(axis=1)]

In [4]:
def create_mapped_json(row):
    result = {}
    skip_cols = list(nifd.columns[nifd.columns.str.contains('CDR')]) + list(nifd.columns[nifd.columns.str.contains('DATE')]) + ['DX', 'CLINICAL_LINKDATE', 'VISIT_NUMBER', 'SITE', 'DOB'] + list(nifd.columns[nifd.columns.str.contains('PSP')]) + list(nifd.columns[nifd.columns.str.contains('PSP')])
    
    for column, value in row.items():
        if column in skip_cols:
            continue
            
        # skip if nan
        if pd.isna(value) or value is None:
            continue
        # convert to number to check if value's negative
        try:
            if float(value) < 0:
                continue  
        except (ValueError, TypeError):
            pass  # Not a number, continue processing
            
        # convert value to appropriate type if specified in dictionary
        if column in data_dict and 'Type' in data_dict[column]:
            try:
                data_type = data_dict[column]['Type']
                if data_type == 'int' or data_type == 'integer':
                    value = int(float(value))
                elif data_type == 'float' or data_type == 'number':
                    value = float(value)
                elif data_type == 'str' or data_type == 'string':
                    # Keep as string, but we already checked for negative values above
                    value = str(value)
                elif data_type == 'bool' or data_type == 'boolean':
                    value = bool(value)
            except (ValueError, TypeError):
                # If conversion fails, skip this value
                continue
            
        if column in data_dict:
            description = data_dict[column]['Description']
            
            # Check if the column has a values key in the dictionary
            if 'Values' in data_dict[column]:
                if str(value) in data_dict[column]['Values']:
                    mapped_value = data_dict[column]['Values'][str(value)]
                    if mapped_value:
                        result[description] = mapped_value
                # Skip this field if value doesn't match any in the dictionary
                else:
                    continue
            # For columns without Values key in dictionary, add the value directly
            else:
                result[description] = value
                
    return json.dumps(result)

In [5]:
nifd['patient_summary'] = nifd.apply(create_mapped_json, axis=1)

In [6]:
def remove_subject_id(json_str):
    data = json.loads(json_str)
    del data['Subject ID']
    return json.dumps(data, indent=4)

# Apply the function to the DataFrame column
nifd["patient_summary"] = nifd["patient_summary"].apply(remove_subject_id)

In [7]:
nifd['DX'].value_counts()


DX
CON                438
BV                 225
SV                 135
PNFA               126
PATIENT (OTHER)     83
L_SD                 8
Name: count, dtype: int64

In [10]:
nifd.to_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd/training_data/NIFD_wjson.csv', index=False)

In [11]:
def get_latest_visits(data):
    result = data.sort_values(by=['LONI_ID', 'VISIT_NUMBER'], ascending=[True, False])
    result = result.drop_duplicates(subset='LONI_ID', keep='first').reset_index(drop=True)
    return result

In [12]:
nifd_latest = get_latest_visits(nifd)

In [13]:
nifd_latest.to_csv('/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd/training_data/train.csv', index=False)